# ⚠️ Risk Scoring & Prioritization (Multi-Layer)

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System     
**Phase:** Phase 4 – Risk Scoring & Prioritization

**Objective:**  
Transform parameter-level normative compliance results into continuous risk scores (0–100)
across multiple regulatory layers (sanitary and discharge), enabling prioritization,
cross-standard comparison (Colombia vs EPA), and decision-oriented analysis without
binary potable/non-potable classification.

## 1. Paths, Data Base and Loads

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "water_quality_analysis.db"
conn = sqlite3.connect(DB_PATH)

# Load catalogs
dim_standard = pd.read_sql_query("SELECT * FROM dim_standard;", conn)
dim_layer = pd.read_sql_query("SELECT * FROM dim_layer;", conn)
dim_parameter = pd.read_sql_query("SELECT * FROM dim_parameter;", conn)

# Compliance table (Phase 3 output)
fact_compliance = pd.read_sql_query("""
SELECT standard_id, CODIGO__MUESTRA, parameter_id, value_num, compliant, deviation
FROM fact_compliance; """, conn)

# Join for readability
dfc = (fact_compliance
        .merge(dim_standard, on="standard_id", how="left")
        .merge(dim_layer, on="layer_id", how="left")
        .merge(dim_parameter, on="parameter_id", how="left"))

display(dfc.head(10))
print("Rows:", len(dfc))
print("Standards:", dfc["standard_code"].unique())
print("Layers:", dfc["layer_code"].unique())

,standard_id,CODIGO__MUESTRA,parameter_id,value_num,compliant,deviation,standard_code,standard_name,layer_id,layer_code,layer_name,parameter_std,canonical_unit
0,3,14615,2,2.60,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,BOD5,mg O2/L
1,3,20148,2,2.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,BOD5,mg O2/L
2,3,22820,2,2.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,BOD5,mg O2/L
3,3,25379,2,5.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,BOD5,mg O2/L
4,3,14615,3,63.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,COD,mg O2/L
5,3,20148,3,11.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,COD,mg O2/L
6,3,22820,3,10.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,COD,mg O2/L
7,3,25379,3,19.00,1,0.0,CO_0631,Colombia Resolution 0631/2015,2,DISCHARGE,Environmental / Discharge pressure,COD,mg O2/L
8,1,14615,13,7.69,1,0.0,CO_2115,Colombia Resolution 2115/2007,1,SANITARY,Sanitary / Drinking-water reference,pH,unidades de pH
9,2,14615,13,7.69,1,0.0,EPA,US EPA Drinking Water (Benchmark),1,SANITARY,Sanitary / Drinking-water reference,pH,unidades de pH


Rows: 48178
Standards: <StringArray>
['CO_0631', 'CO_2115', 'EPA']
Length: 3, dtype: str
Layers: <StringArray>
['DISCHARGE', 'SANITARY']
Length: 2, dtype: str


## 2. Risk Model (Continuous 0–100)

Risk is computed as a continuous score per sample and layer based on:

1) **Non-compliance frequency** (how many parameters fail)
2) **Non-compliance severity** (how far results exceed limits)

This design avoids binary potable/not-potable labels and supports prioritization.

In [2]:
# Parameter weights (editable)
WEIGHTS = { # Sanitary (Layer 1)
            "pH": 1.5,
            "Turbidity": 2.0,

            # Discharge (Layer 2)
            "BOD5": 2.0,
            "COD": 2.0,
            "Total Suspended Solids": 1.5,
            "Temperature": 1.0,}

dfc["w"] = dfc["parameter_std"].map(WEIGHTS).fillna(1.0)

# Normalize severity-
# deviation: 0 if compliant, >0 if exceeds limit (or out of range)
# We need a bounded severity so one extreme value doesn't dominate.
# severity_norm in [0,1], using a smooth transform:
#   severity_norm = deviation / (deviation + k)
# where k controls how fast it saturates (bigger k = less sensitive).
k = 5.0

dfc["deviation"] = dfc["deviation"].fillna(0.0)
dfc["severity_norm"] = dfc["deviation"] / (dfc["deviation"] + k)

# Also define a "violation" indicator
dfc["violation"] = (dfc["compliant"] == 0).astype(int)

display(dfc[["standard_code","layer_code","parameter_std","value_num","compliant","deviation","severity_norm","w"]].head(15))

,standard_code,layer_code,parameter_std,value_num,compliant,deviation,severity_norm,w
0,CO_0631,DISCHARGE,BOD5,2.60,1,0.00,0.000000,2.0
1,CO_0631,DISCHARGE,BOD5,2.00,1,0.00,0.000000,2.0
2,CO_0631,DISCHARGE,BOD5,2.00,1,0.00,0.000000,2.0
3,CO_0631,DISCHARGE,BOD5,5.00,1,0.00,0.000000,2.0
4,CO_0631,DISCHARGE,COD,63.00,1,0.00,0.000000,2.0
5,CO_0631,DISCHARGE,COD,11.00,1,0.00,0.000000,2.0
6,CO_0631,DISCHARGE,COD,10.00,1,0.00,0.000000,2.0
7,CO_0631,DISCHARGE,COD,19.00,1,0.00,0.000000,2.0
8,CO_2115,SANITARY,pH,7.69,1,0.00,0.000000,1.5
9,EPA,SANITARY,pH,7.69,1,0.00,0.000000,1.5


## 3. Layer Risk Scores

For each sample and standard:

- **Violation rate score (0–100):** weighted share of failed parameters
- **Severity score (0–100):** weighted average of normalized deviations
- **Risk score (0–100):** combined score

A critical flag is triggered if any high-impact parameter fails.

In [3]:
# Critical parameters per layer (editable)
CRITICAL_PARAMS_SANITARY = {"Turbidity", "pH"} 
CRITICAL_PARAMS_DISCHARGE = {"BOD5", "COD", "Total Suspended Solids"}

def is_critical(row):
    if row["layer_code"] == "SANITARY":
        return int((row["parameter_std"] in CRITICAL_PARAMS_SANITARY) and (row["compliant"] == 0))
    if row["layer_code"] == "DISCHARGE":
        return int((row["parameter_std"] in CRITICAL_PARAMS_DISCHARGE) and (row["compliant"] == 0))
    return 0

dfc["critical_hit"] = dfc.apply(is_critical, axis=1)

# Aggregate to sample-standard-layer
grp_cols = ["standard_id","standard_code","layer_id","layer_code","CODIGO__MUESTRA"]

agg = (dfc.groupby(grp_cols)
        .apply(lambda g: pd.Series({
            "n_params": len(g),
            "w_sum": g["w"].sum(),
           "violation_weighted_rate": (g["w"] * g["violation"]).sum() / (g["w"].sum() if g["w"].sum() else 1.0),
           "severity_weighted_avg": (g["w"] * g["severity_norm"]).sum() / (g["w"].sum() if g["w"].sum() else 1.0),
            "critical_violation_flag": int(g["critical_hit"].max() == 1),
        }))
        .reset_index())

# Convert to 0–100 scales
agg["violation_score_0_100"] = (agg["violation_weighted_rate"] * 100).round(2)
agg["severity_score_0_100"] = (agg["severity_weighted_avg"] * 100).round(2)

# Combine (tunable weights)
ALPHA = 0.7  # weight on violations
BETA  = 0.3  # weight on severity

agg["risk_score_0_100"] = (ALPHA*agg["violation_score_0_100"] + BETA*agg["severity_score_0_100"]).round(2)

display(agg.head(20))

# Quick sanity: see worst samples per standard/layer
display(agg.sort_values("risk_score_0_100", ascending=False)
        .head(20)[["standard_code","layer_code","CODIGO__MUESTRA","n_params","risk_score_0_100","critical_violation_flag"]])

,standard_id,standard_code,layer_id,layer_code,CODIGO__MUESTRA,n_params,w_sum,violation_weighted_rate,severity_weighted_avg,critical_violation_flag,violation_score_0_100,severity_score_0_100,risk_score_0_100
0,1,CO_2115,1,SANITARY,11284,2.0,3.5,0.571429,0.214286,1.0,57.14,21.43,46.43
1,1,CO_2115,1,SANITARY,11285,2.0,3.5,0.571429,0.469388,1.0,57.14,46.94,54.08
2,1,CO_2115,1,SANITARY,11289,2.0,3.5,1.000000,0.555390,1.0,100.00,55.54,86.66
3,1,CO_2115,1,SANITARY,11291,2.0,3.5,0.571429,0.507937,1.0,57.14,50.79,55.24
4,1,CO_2115,1,SANITARY,11292,2.0,3.5,0.571429,0.492063,1.0,57.14,49.21,54.76
5,1,CO_2115,1,SANITARY,11294,2.0,3.5,0.571429,0.554622,1.0,57.14,55.46,56.64
6,1,CO_2115,1,SANITARY,11297,2.0,3.5,0.571429,0.566586,1.0,57.14,56.66,57.00
7,1,CO_2115,1,SANITARY,11298,2.0,3.5,0.571429,0.562444,1.0,57.14,56.24,56.87
8,1,CO_2115,1,SANITARY,11299,2.0,3.5,0.571429,0.507937,1.0,57.14,50.79,55.24
9,1,CO_2115,1,SANITARY,11300,2.0,3.5,0.571429,0.561404,1.0,57.14,56.14,56.84


,standard_code,layer_code,CODIGO__MUESTRA,n_params,risk_score_0_100,critical_violation_flag
9095,EPA,SANITARY,18177,1.0,99.96,1.0
2280,CO_2115,SANITARY,18177,1.0,99.96,1.0
7354,EPA,SANITARY,13741,1.0,99.88,1.0
539,CO_2115,SANITARY,13741,1.0,99.88,1.0
7096,EPA,SANITARY,12617,1.0,99.85,1.0
281,CO_2115,SANITARY,12617,1.0,99.85,1.0
7761,EPA,SANITARY,14932,1.0,99.76,1.0
7762,EPA,SANITARY,14933,1.0,99.76,1.0
947,CO_2115,SANITARY,14933,1.0,99.76,1.0
946,CO_2115,SANITARY,14932,1.0,99.76,1.0


## 4. Store Risk Outputs (SQL + CSV)

This section stores layer-level risk scores in the database and exports a clean CSV
for downstream reporting and dashboards.

In [4]:
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS fact_layer_risk;

CREATE TABLE fact_layer_risk (
  standard_id INTEGER,
  layer_id INTEGER,
  CODIGO__MUESTRA INTEGER,
  n_params INTEGER,
  violation_score_0_100 REAL,
  severity_score_0_100 REAL,
  risk_score_0_100 REAL,
  critical_violation_flag INTEGER,
  created_at TEXT DEFAULT (datetime('now')),
  PRIMARY KEY (standard_id, layer_id, CODIGO__MUESTRA)
);
""")
conn.commit()

to_insert = agg[[
    "standard_id","layer_id","CODIGO__MUESTRA","n_params",
    "violation_score_0_100","severity_score_0_100","risk_score_0_100",
    "critical_violation_flag"
]].copy()

cur.executemany("""
INSERT OR REPLACE INTO fact_layer_risk
(standard_id, layer_id, CODIGO__MUESTRA, n_params,
 violation_score_0_100, severity_score_0_100, risk_score_0_100,
 critical_violation_flag)
VALUES (?, ?, ?, ?, ?, ?, ?, ?);
""", to_insert.itertuples(index=False, name=None))

conn.commit()

# Verify table
check = pd.read_sql_query("""
SELECT r.*, s.standard_code, l.layer_code
FROM fact_layer_risk r
JOIN dim_standard s ON r.standard_id = s.standard_id
JOIN dim_layer l ON r.layer_id = l.layer_id
ORDER BY r.risk_score_0_100 DESC
LIMIT 20;
""", conn)
display(check)

# Export deliverable CSV
OUT_CSV = OUTPUT_DIR / "phase4_layer_risk_scores.csv"
check_all = pd.read_sql_query("""
SELECT r.*, s.standard_code, l.layer_code
FROM fact_layer_risk r
JOIN dim_standard s ON r.standard_id = s.standard_id
JOIN dim_layer l ON r.layer_id = l.layer_id;
""", conn)
check_all.to_csv(OUT_CSV, index=False)

print("✅ Phase 4 risk outputs exported to:", OUT_CSV)

,standard_id,layer_id,CODIGO__MUESTRA,n_params,violation_score_0_100,severity_score_0_100,risk_score_0_100,critical_violation_flag,created_at,standard_code,layer_code
0,1,1,18177,1,100.0,99.85,99.96,1,2026-02-25 17:23:54,CO_2115,SANITARY
1,2,1,18177,1,100.0,99.85,99.96,1,2026-02-25 17:23:54,EPA,SANITARY
2,1,1,13741,1,100.0,99.59,99.88,1,2026-02-25 17:23:54,CO_2115,SANITARY
3,2,1,13741,1,100.0,99.59,99.88,1,2026-02-25 17:23:54,EPA,SANITARY
4,1,1,12617,1,100.0,99.50,99.85,1,2026-02-25 17:23:54,CO_2115,SANITARY
5,2,1,12617,1,100.0,99.50,99.85,1,2026-02-25 17:23:54,EPA,SANITARY
6,1,1,14932,1,100.0,99.20,99.76,1,2026-02-25 17:23:54,CO_2115,SANITARY
7,1,1,14933,1,100.0,99.20,99.76,1,2026-02-25 17:23:54,CO_2115,SANITARY
8,2,1,14932,1,100.0,99.19,99.76,1,2026-02-25 17:23:54,EPA,SANITARY
9,2,1,14933,1,100.0,99.19,99.76,1,2026-02-25 17:23:54,EPA,SANITARY


✅ Phase 4 risk outputs exported to: D:\Documents\Portfolio\01-water-quality-normative\outputs\phase4_layer_risk_scores.csv


## 5. Sanitary Risk Comparison: Colombia (Resolution 2115) vs EPA Benchmarks


In [5]:
# Load layer risk scores (sanitary layer only)
risk = pd.read_sql_query("""
SELECT
    r.CODIGO__MUESTRA,
    r.risk_score_0_100,
    r.critical_violation_flag,
    s.standard_code,
    l.layer_code
FROM fact_layer_risk r
JOIN dim_standard s ON r.standard_id = s.standard_id
JOIN dim_layer l ON r.layer_id = l.layer_id
WHERE l.layer_code = 'SANITARY'
""", conn)

display(risk.head())

# Pivot: one row per sample, one column per standard
risk_pivot = (
    risk.pivot_table(
        index="CODIGO__MUESTRA",
        columns="standard_code",
        values="risk_score_0_100")
    .reset_index())

display(risk_pivot.head())

# Compute comparative indicators

risk_pivot["epa_minus_colombia"] = (
    risk_pivot["EPA"] - risk_pivot["CO_2115"])

risk_pivot["epa_stricter_flag"] = (
    risk_pivot["epa_minus_colombia"] > 0).astype(int)

risk_pivot["colombia_only_acceptable"] = ((risk_pivot["CO_2115"] < 20) &
                                        (risk_pivot["EPA"] >= 20)).astype(int)

display(risk_pivot.sort_values("epa_minus_colombia", ascending=False).head(20))

# Summary statistics
summary = pd.Series({
    "Total samples compared": len(risk_pivot),
    "EPA stricter than Colombia (%)":
        round(risk_pivot["epa_stricter_flag"].mean() * 100, 2),
    "Colombia acceptable / EPA risky (%)":
        round(risk_pivot["colombia_only_acceptable"].mean() * 100, 2),
    "Mean risk score (CO_2115)":
        round(risk_pivot["CO_2115"].mean(), 2),
    "Mean risk score (EPA)":
        round(risk_pivot["EPA"].mean(), 2),})

display(summary)

# Export comparison table
OUT_CSV = OUTPUT_DIR / "phase4_sanitary_risk_comparison_CO_vs_EPA.csv"
risk_pivot.to_csv(OUT_CSV, index=False)

print("✅ Colombia vs EPA comparison exported to:")
print(OUT_CSV)

,CODIGO__MUESTRA,risk_score_0_100,critical_violation_flag,standard_code,layer_code
0,11284,46.43,1,CO_2115,SANITARY
1,11285,54.08,1,CO_2115,SANITARY
2,11289,86.66,1,CO_2115,SANITARY
3,11291,55.24,1,CO_2115,SANITARY
4,11292,54.76,1,CO_2115,SANITARY


standard_code,CODIGO__MUESTRA,CO_2115,EPA
0,11284,46.43,0.00
1,11285,54.08,53.71
2,11289,86.66,87.17
3,11291,55.24,55.10
4,11292,54.76,54.54


standard_code,CODIGO__MUESTRA,CO_2115,EPA,epa_minus_colombia,epa_stricter_flag,colombia_only_acceptable
5508,27815,0.00,71.43,71.43,1,1
6750,29980,0.00,70.76,70.76,1,1
4322,24164,0.00,70.53,70.53,1,1
2891,19832,56.61,87.75,31.14,1,0
3036,20262,57.13,88.15,31.02,1,0
4694,25205,56.50,87.52,31.02,1,0
6658,29847,0.00,30.98,30.98,1,1
6148,29021,55.97,86.85,30.88,1,0
859,14746,56.85,87.71,30.86,1,0
506,13625,55.15,85.98,30.83,1,0


Total samples compared                 6815.00
EPA stricter than Colombia (%)            1.13
Colombia acceptable / EPA risky (%)       0.26
Mean risk score (CO_2115)                55.64
Mean risk score (EPA)                    52.84
dtype: float64

✅ Colombia vs EPA comparison exported to:
D:\Documents\Portfolio\01-water-quality-normative\outputs\phase4_sanitary_risk_comparison_CO_vs_EPA.csv


## Phase 4 Summary

This phase transformed parameter-level compliance into continuous risk scores
(0–100) across sanitary and discharge regulatory layers.

A comparative analysis between Colombian drinking-water regulations (Resolution 2115)
and EPA benchmarks revealed systematic differences in regulatory strictness,
highlighting samples that are acceptable locally but present higher risk under
international standards.

This risk-based, multi-layer framework supports prioritization, regulatory comparison,
and advanced decision-making without relying on binary classifications.